# Archived research notebook

Original filename: `preprocessing second 全转fif.ipynb`  
Purpose: Convert EDF recordings to FIF.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
import os
import mne
from collections import defaultdict
from datetime import datetime


# 解析摘要文件获取癫痫发作信息
def parse_seizure_summary(summary_path):
    seizure_info = defaultdict(list)
    current_file = None

    with open(summary_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith('File Name:'):
                file_name = line.split(': ')[1].split('.edf')[0] + '.edf'
                current_file = file_name
            elif line.startswith('Seizure Start Time:') and current_file:
                start_sec = float(line.split(': ')[1].split(' seconds')[0])
            elif line.startswith('Seizure End Time:') and current_file:
                end_sec = float(line.split(': ')[1].split(' seconds')[0])
                seizure_info[current_file].append((start_sec, end_sec))

    return seizure_info


# 将无发作的 EDF 文件转换为 FIF 文件
def convert_non_seizure_edf_to_fif(folder_path, summary_path):
    seizure_info = parse_seizure_summary(summary_path)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.edf'):
                edf_path = os.path.join(root, file)
                edf_name = file
                seizures = seizure_info.get(edf_name, [])

                if seizures:
                    print(f"跳过 {edf_name}：有发作信息")
                    continue

                try:
                    # 读取 EDF 文件（preload=True 确保注释正确关联时间）
                    raw = mne.io.read_raw_edf(edf_path, preload=True)

                    # -------------------- 处理 datetime 类型的 meas_date --------------------
                    meas_date = raw.info.get('meas_date')
                    if isinstance(meas_date, datetime):
                        # 将 datetime 对象转换为 Unix 时间戳（整数秒）
                        meas_date = int(meas_date.timestamp())
                        print(f"检测到 datetime 时间戳，转换为 {meas_date}")
                    elif isinstance(meas_date, (float, int)):
                        meas_date = int(meas_date)  # 确保为整数
                    else:
                        meas_date = 0  # 其他情况设为0

                    # 校验时间戳是否在有效范围内
                    if not (-2147483648 <= meas_date <= 2147483647):
                        print(f"重置无效时间戳 {meas_date} 为 0")
                        meas_date = 0
                    raw.set_meas_date(meas_date)
                    # -------------------- 其他逻辑保持不变 --------------------

                    # 保存为符合 MNE 规范的文件名
                    fif_name = edf_name.replace('.edf', '_raw.fif')
                    fif_path = os.path.join(root, fif_name)
                    raw.save(fif_path, overwrite=True)
                    print(f"已将 {edf_name} 转换并保存为 {os.path.basename(fif_path)}")

                except Exception as e:
                    print(f"处理 {edf_name} 时出错: {e}")


# 主函数
if __name__ == "__main__":
    folder_path = r"data/chb-mit/chb01"  # EDF 文件路径
    summary_path = r"data/chb-mit/chb01/chb01-summary.txt"  # 摘要文件路径

    convert_non_seizure_edf_to_fif(folder_path, summary_path)
    print("所有文件处理完成")